In [14]:
from __future__ import annotations

#TARGET_MODEL = "mistralai/Mistral-7B-v0.1"
#TARGET_MODEL = "HuggingFaceH4/zephyr-7b-alpha"
TARGET_MODEL = "microsoft/deberta-v3-large"

DEBUG = False

In [15]:
# %% Directory settings

# ====================================================
# Directory settings
# ====================================================
from pathlib import Path

OUTPUT_DIR = Path("./")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

INPUT_DIR = Path("../input/")

In [16]:
import pandas as pd

# reference: https://www.kaggle.com/datasets/thedrcat/daigt-proper-train-dataset
train_datas=pd.read_csv("/kaggle/input/daigt-proper-train-dataset/train_drcat_04.csv")
print(f'daigt-external-dataset: {train_datas.shape}')
trains_data=train_datas[train_datas["label"]==1] # .sample(3000) delete samples, reference: https://www.kaggle.com/code/pamin2222/votingclf-alldata-modweight-0-899
# trains_data_real=train_datas[train_datas["label"]==0]
trains_data=pd.concat([trains_data,trains_data,trains_data])  # TODO, reference:
trains_data=trains_data[["text","label"]]
trains_data['text'] = trains_data['text'].str.replace('\n', '')
print(f'daigt-external-dataset（after）: {trains_data.shape}')
display(trains_data.head())

# reference: https://www.kaggle.com/datasets/alejopaullier/daigt-external-dataset
external_df = pd.read_csv("/kaggle/input/daigt-external-dataset/daigt_external_dataset.csv", sep=',')
print(f'daigt-external-dataset: {external_df.shape}')
external_df = external_df.rename(columns={'generated': 'label'})
external_df = external_df[["source_text"]]
external_df.columns = ["text"]
external_df['text'] = external_df['text'].str.replace('\n', '')
external_df["label"] = 1
external_df = pd.concat([external_df,external_df,external_df]) # TODO
print(f'daigt-external-dataset(after): {external_df.shape}')

train = pd.read_csv("/kaggle/input/daigt-v2-train-dataset/train_v2_drcat_02.csv")
train = train[train.RDizzl3_seven == True].reset_index(drop=True)
print(f'daigt-v2-train-dataset: {train.shape}')

train=pd.concat([train,external_df,trains_data])
print(train.value_counts("label"))
train.reset_index(inplace=True, drop=True)

daigt-external-dataset: (44206, 6)
daigt-external-dataset（after）: (43242, 2)


,text,label
0,"In recent years, technology has had a profoun...",1
4,I strongly believe that meditation and mindful...,1
9,One way school administrators can attempt to c...,1
11,While summer is meant as a break from the regu...,1
12,The use of Facial Action Coding System (FACS) ...,1


daigt-external-dataset: (2421, 4)
daigt-external-dataset(after): (7263, 2)
daigt-v2-train-dataset: (20450, 5)
label
1    56705
0    14250
Name: count, dtype: int64


In [17]:
print(train.shape)
display(train.head())

(70955, 5)


,text,label,prompt_name,source,RDizzl3_seven
0,Cars have been around for awhile and they have...,0,Car-free cities,persuade_corpus,True
1,Have you ever thought what it would be like no...,0,Car-free cities,persuade_corpus,True
2,What you are about to read is going to give yo...,0,Car-free cities,persuade_corpus,True
3,cars have many flaws nd and in this day and ag...,0,Car-free cities,persuade_corpus,True
4,There are many advantages of limiting car usag...,0,Car-free cities,persuade_corpus,True


In [18]:
print(train.label.value_counts(),"\n")
print(train.source.value_counts())

label
1    56705
0    14250
Name: count, dtype: int64 

source
persuade_corpus                       12875
kingki19_palm                          1384
train_essays                           1378
radek_500                               500
llama_70b_v1                            499
darragh_claude_v6                       481
darragh_claude_v7                       469
falcon_180b_v1                          417
NousResearch/Llama-2-7b-chat-hf         400
mistralai/Mistral-7B-Instruct-v0.1      399
cohere-command                          349
palm-text-bison1                        348
llama2_chat                             277
radekgpt4                               200
mistral7binstruct_v1                    188
chat_gpt_moth                           159
mistral7binstruct_v2                    127
Name: count, dtype: int64


In [19]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X = train.loc[:, train.columns != "label"]
y = train.loc[:, train.columns == "label"]

for i, (train_index, valid_index) in enumerate(skf.split(X, y)):
    train.loc[valid_index, "fold"] = i
    
print(train.groupby("fold")["label"].value_counts())
train.head()

fold  label
0.0   1        11341
      0         2850
1.0   1        11341
      0         2850
2.0   1        11341
      0         2850
3.0   1        11341
      0         2850
4.0   1        11341
      0         2850
Name: count, dtype: int64


,text,label,prompt_name,source,RDizzl3_seven,fold
0,Cars have been around for awhile and they have...,0,Car-free cities,persuade_corpus,True,0.0
1,Have you ever thought what it would be like no...,0,Car-free cities,persuade_corpus,True,1.0
2,What you are about to read is going to give yo...,0,Car-free cities,persuade_corpus,True,1.0
3,cars have many flaws nd and in this day and ag...,0,Car-free cities,persuade_corpus,True,4.0
4,There are many advantages of limiting car usag...,0,Car-free cities,persuade_corpus,True,4.0


In [20]:
train.drop(columns=["source", "prompt_name", "RDizzl3_seven"],inplace=True)

print(train.shape)
display(train.head())

(70955, 3)


,text,label,fold
0,Cars have been around for awhile and they have...,0,0.0
1,Have you ever thought what it would be like no...,0,1.0
2,What you are about to read is going to give yo...,0,1.0
3,cars have many flaws nd and in this day and ag...,0,4.0
4,There are many advantages of limiting car usag...,0,4.0


In [21]:
train["fold"] = train["fold"].astype(int)
print(train.fold.value_counts())

fold
0    14191
1    14191
4    14191
3    14191
2    14191
Name: count, dtype: int64


In [22]:
# fold0 as valid
val = train[train["fold"] == 0]
train = train[train["fold"] != 0]

In [23]:
display(train.head(3))
display(val.head(3))

,text,label,fold
1,Have you ever thought what it would be like no...,0,1
2,What you are about to read is going to give yo...,0,1
3,cars have many flaws nd and in this day and ag...,0,4


,text,label,fold
0,Cars have been around for awhile and they have...,0,0
6,"Here in the United States, we have a problem o...",0,0
8,Have you ever walked a certain number of miles...,0,0


In [24]:
train_df = train[["text", "label"]]
valid_df = val[["text", "label"]]
print(train_df.shape)
print(train_df.label.value_counts())
print(valid_df.shape)
print(valid_df.label.value_counts())

(56764, 2)
label
1    45364
0    11400
Name: count, dtype: int64
(14191, 2)
label
1    11341
0     2850
Name: count, dtype: int64


In [ ]:
!pip install -q -U peft --no-index --find-links ../input/llm-detect-pip/
!pip install -q -U accelerate --no-index --find-links ../input/llm-detect-pip/
!pip install -q -U bitsandbytes --no-index --find-links ../input/llm-detect-pip/
!pip install -q -U transformers --no-index --find-links ../input/llm-detect-pip/

In [ ]:
# load model with 4bit bnb

from peft import get_peft_config, PeftModel, PeftConfig, get_peft_model, LoraConfig, TaskType # type: ignore
from transformers import BitsAndBytesConfig
import torch

peft_config = LoraConfig(
    r=64,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    target_modules=[
        "q_proj",
        "v_proj"
    ],
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

#bnb_config = BitsAndBytesConfig(
#    load_in_8bit=True,
#    bnb_8bit_quant_type="nf8",
#    bnb_8bit_use_double_quant=True,
#    bnb_8bit_compute_dtype=torch.float16
#)

In [ ]:
from transformers import AutoTokenizer, LlamaForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
base_model = LlamaForSequenceClassification.from_pretrained(
    TARGET_MODEL,
    num_labels=2,
    quantization_config=bnb_config,   #TO QUANTIZE
    device_map={"":0}
)
base_model.config.pretraining_tp = 1 # 1 is 7b
base_model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
model = get_peft_model(base_model, peft_config)

In [ ]:
model.print_trainable_parameters()

In [ ]:
print(train_df.shape)
print(valid_df.shape)

In [ ]:
print(train_df.label.value_counts(), valid_df.label.value_counts())

In [ ]:
# datasets
from datasets import Dataset

# from pandas
train_ds = Dataset.from_pandas(train_df)
valid_ds = Dataset.from_pandas(valid_df)

In [ ]:
def preprocess_function(examples, max_length=512):   
    return tokenizer(examples["text"], truncation=True, max_length=max_length, padding=True)

In [ ]:
train_tokenized_ds = train_ds.map(preprocess_function, batched=True)
valid_tokenized_ds = valid_ds.map(preprocess_function, batched=True)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy_val = accuracy_score(labels, predictions)
    roc_auc_val = roc_auc_score(labels, predictions)
    
    return {
        "accuracy": accuracy_val,
        "roc_auc": roc_auc_val,
    }

In [ ]:
#slower train?
#5 fold cv?
#less quantized?
#ensemble deberta-v3-large, mistral 7B, and zephyr 7B alpha

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers import EarlyStoppingCallback

steps = 5 if DEBUG else 20

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=5e-5,   #5e-6
    per_device_train_batch_size=1,   #2
    per_device_eval_batch_size=1,    #2
    gradient_accumulation_steps=16,
    max_grad_norm=0.3,
    optim='paged_adamw_32bit',
    lr_scheduler_type="cosine",
    num_train_epochs=1,
    weight_decay=0.01,             #0.01 was original
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    push_to_hub=False,
    warmup_steps=steps,
    eval_steps=steps,
    logging_steps=steps,
    report_to='none'
)

early_stopping = EarlyStoppingCallback(early_stopping_patience=2)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized_ds,
    eval_dataset=valid_tokenized_ds,
    tokenizer=tokenizer,
    callbacks=[early_stopping],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
from shutil import rmtree

trainer.save_model(output_dir=str(OUTPUT_DIR))

for path in Path(training_args.output_dir).glob("checkpoint-*"):
    if path.is_dir():
        rmtree(path)

In [ ]:
del trainer, model, base_model

In [ ]:
# cuda cache clear
import torch
torch.cuda.empty_cache()

In [ ]:
# load model / tokenizer with 4bit bnb

from peft import get_peft_config, PeftModel, PeftConfig, get_peft_model, LoraConfig, TaskType # type: ignore
from transformers import BitsAndBytesConfig
import torch


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
from transformers import AutoTokenizer, LlamaForSequenceClassification

base_model = LlamaForSequenceClassification.from_pretrained(
    TARGET_MODEL,
    num_labels=2,
    quantization_config=bnb_config,
    device_map={"":0}
)
base_model.config.pretraining_tp = 1 # 1 is 7b
base_model.config.pad_token_id = tokenizer.pad_token_id


In [ ]:
model = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR))


In [ ]:
trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

In [ ]:
pred_output = trainer.predict(valid_tokenized_ds)
logits = pred_output.predictions

In [ ]:
logits = pred_output.predictions

In [ ]:
valid_df.label

In [ ]:
logits

In [ ]:
# from scipy.special import expit as sigmoid
import numpy as np
def sigmoid(x):
    return 1 / (1 + np.exp(-x))  
probs = sigmoid(logits[:, 1])
probs.shape, probs[0:5]

In [ ]:
sub = pd.DataFrame()
sub['id'] = valid_df['id']
sub['generated'] = probs
# sub.to_csv('submission.csv', index=False)
sub.head()

In [ ]:
!ls -alh /kaggle/working/